# Create representative Proteome-Constrained RBC model for dataset
## Setup
### Import packages

In [2]:
import itertools
from collections import defaultdict
from pathlib import Path

import gurobipy as gp
import pandas as pd
from rbc_gem_utils import (COBRA_CONFIGURATION, DATABASE_PATH, GEM_NAME,
                           PROCESSED_PATH, ROOT_PATH, build_string,
                           get_annotation_df, read_cobra_model, show_versions,
                           split_string, write_cobra_model)
from rbc_gem_utils.analysis.overlay import (
    ATTR_SUBCLASS_DICT, DEFAULT_COMPARTMENT_CONSTRAINT_PREFIX,
    DEFAULT_CONCENTRATION_BOUND, DEFAULT_CONSTRAINT_PREFIX,
    DEFAULT_ENZYME_FORWARD_SUFFIX, DEFAULT_ENZYME_REVERSE_SUFFIX,
    DEFAULT_ENZYME_TOTAL_SUFFIX, DEFAULT_ISOFORM_CONSTRAINT_PREFIX,
    DEFAULT_KEFF, ComplexDilution, Enzyme, EnzymeDilution, Protein,
    ProteinDilution, ProteomeBudget, ProteomeBudgetDilution,
    add_relaxation_budget, construct_pcmodel_from_tables, create_complex_table,
    create_enzyme_table, create_protein_table, create_sequence_table,
    load_overlay_model)
from rbc_gem_utils.database.uniprot import UNIPROT_DB_TAG, UNIPROT_PATH
from rbc_gem_utils.util import convert_gDW_to_L, convert_L_to_gDW, strip_plural

gp.setParam("OutputFlag", 0)
gp.setParam("LogToConsole", 0)

# Show versions of notebook
show_versions()

Set parameter Username

Package Information
-------------------
rbc-gem-utils 0.0.1

Dependency Information
----------------------
beautifulsoup4                       4.12.3
bio                                 missing
cobra                                0.29.1
depinfo                               2.2.0
gurobipy                             11.0.3
matplotlib                           3.10.0
matplotlib-venn                       1.1.1
memote                               0.17.0
networkx                              3.4.2
notebook                              7.3.2
openpyxl                              3.1.5
pandas                                2.2.3
pre-commit                            4.1.0
rbc-gem-utils[database,network,vis] missing
requests                             2.32.3
scipy                                1.15.1
seaborn                              0.13.2

Build Tools Information
-----------------------
pip          24.2
setuptools 75.1.0
wheel      0.44.0

Platform Informat

### Define configuration
#### COBRA Configuration

In [3]:
COBRA_CONFIGURATION.solver = "gurobi"
COBRA_CONFIGURATION.bounds = (-1e8, 1e8)
COBRA_CONFIGURATION

Attribute,Description,Value
solver,Mathematical optimization solver,gurobi
tolerance,"General solver tolerance (feasibility, integrality, etc.)",1e-07
lower_bound,Default reaction lower bound,-100000000.0
upper_bound,Default reaction upper bound,100000000.0
processes,Number of parallel processes,127
cache_directory,Path for the model cache,C:\Users\Alicia Key\AppData\Local\opencobra\cobrapy\Cache
max_cache_size,Maximum cache size in bytes,104857600
cache_expiration,Model cache expiration time in seconds (if any),None


## Load RBC model

In [4]:
# model_id = "RBC_GEM"

# data_path = Path(f"{ROOT_PATH}/data/analysis/OVERLAY").resolve()
# results_path = Path(f"{ROOT_PATH}{PROCESSED_PATH}/{model_id}/OVERLAY")
# results_path.mkdir(exist_ok=True, parents=True)

# imagetype = "svg"
# transparent = True
# save_figures = True

# dataset_name = "RBComics"
# pcmodel_dirpath = Path(f"{data_path}/{model_id}")
# dataset_path = Path(f"{results_path}/{dataset_name}")
# dataset_models_dirpath = Path(f"{dataset_path}/pcmodels")

# sample_prefix, time_prefix = ("S", "D")
# # Integers are easier to work with for time points
# timepoints = [10, 23, 42]

# model = read_cobra_model(filename=f"{pcmodel_dirpath}/{model_id}.xml")
# pcmodel = load_overlay_model(filename=f"{pcmodel_dirpath}/{model_id}_PC.xml")

# # For this workflow, shut off complex dilution reactions at the start
# for cplx_dilution in pcmodel.reactions.query(lambda x: isinstance(x, ComplexDilution)):
#     cplx_dilution.bounds = (0, 0)

# pcmodel

model_id = "RBC3P"

data_path = Path("../../../../data/analysis/OVERLAY").resolve()
root_path = Path("../../../..").resolve()
results_path = root_path / "data" / "processed" / model_id / "OVERLAY"
results_path.mkdir(exist_ok=True, parents=True)

imagetype = "svg"
transparent = True
save_figures = True

dataset_name = "RBComics"
pcmodel_dirpath = data_path / model_id
dataset_path = results_path / dataset_name
dataset_models_dirpath = dataset_path / "pcmodels"

sample_prefix, time_prefix = ("S", "D")
# Integers are easier to work with for time points
timepoints = [10, 23, 42]

model_filename = pcmodel_dirpath / f"{model_id}.xml"
model = read_cobra_model(filename=model_filename)
pcmodel_filename = pcmodel_dirpath / f"{model_id}_PC.xml"
pcmodel = load_overlay_model(filename=pcmodel_filename)

# For this workflow, shut off complex dilution reactions at the start
for cplx_dilution in pcmodel.reactions.query(lambda x: isinstance(x, ComplexDilution)):
    cplx_dilution.bounds = (0, 0)

pcmodel

Name,RBC3P_PC
Memory address,1a953d6d580
Number of metabolites,489
Number of reactions,935
Number of genes,103
Number of groups,10
Objective expression,1.0*NaKt - 1.0*NaKt_reverse_db47e
Compartments,"cytosol, extracellular space, protein compartment"


### Create PC-model representative of dataset
Can only be done after pcFVA results are generated

In [5]:
sample_model = load_overlay_model(filename=dataset_models_dirpath / "RBC3P_PC_Mean_D10.xml")
sample_model.optimize()

,fluxes,reduced_costs
ADA,0.0,0.0
ADEt,0.0,0.0
ADK1,0.0,-0.0
ADNK1,0.0,-2.0
ADNt,0.0,0.0
...,...,...
RELAX_protein_TMEM109_pc,0.0,0.0
RELAX_protein_TPI1_pc,0.0,0.0
RELAX_protein_TRPC6_pc,0.0,0.0
RELAX_protein_TRPV2_pc,0.0,0.0


In [17]:
df_reaction_ranges_filename = dataset_path / f"{pcmodel.id}_{dataset_name}_reaction_bounds.tsv"
df_reaction_bounds = pd.read_csv(
    df_reaction_ranges_filename,
    sep="\t",
    index_col="reactions",
)
df_reaction_bounds = df_reaction_bounds.rename(
    {"minimum": "lower_bound", "maximum": "upper_bound"}, axis=1
)
df_reaction_bounds

,lower_bound,upper_bound
reactions,,
ADA,0.000000e+00,4.436926e+00
ADEt,-4.654455e-01,1.700029e-16
ADK1,-1.110223e-16,4.654455e-01
ADNK1,0.000000e+00,4.654455e-01
ADNt,-4.266844e+00,4.654455e-01
...,...,...
SPODM,0.000000e+00,2.059212e+00
TALA,-7.301483e-01,6.959665e-01
TKT1,-7.301483e-01,6.959665e-01


In [7]:
pcmodel_dataset_parameterized = pcmodel.copy()
pcmodel_dataset_parameterized.id += f"_{dataset_name}"
add_relaxation_budget(pcmodel_dataset_parameterized, 0, verbose=False)
for rid, bounds in df_reaction_bounds.iterrows():
    reaction = pcmodel_dataset_parameterized.reactions.get_by_id(rid)
    reaction.bounds = bounds

original_relaxation_bounds = pcmodel_dataset_parameterized.reactions.get_by_id(
    "PBDL_relaxation_budget"
).bounds
# Protein constrained  without curated keffs
cobra_model_filename = pcmodel_dirpath / f"{pcmodel_dataset_parameterized}.xml"
write_cobra_model(
    pcmodel_dataset_parameterized, filename=cobra_model_filename
)
pcmodel_dataset_parameterized

Name,RBC3P_PC_RBComics
Memory address,1a954388aa0
Number of metabolites,490
Number of reactions,1039
Number of genes,103
Number of groups,10
Objective expression,1.0*NaKt - 1.0*NaKt_reverse_db47e
Compartments,"cytosol, extracellular space, protein compartment"


In [16]:
pcmodel_dataset_parameterized.reactions.get_by_id("PBDL_relaxation_budget").upper_bound = 10.0
for reaction in pcmodel_dataset_parameterized.reactions:
    if "budget" in reaction.id.lower() or "relax" in reaction.id.lower():
        print(reaction.id, reaction.bounds)
pcmodel_dataset_parameterized.optimize()

PBDL_proteome_budget (0.0, 100.0)
PBDL_hemoglobin_budget (900.0, 1000.0)
PBDL_total_budget (0.0, 1000.0)
PBDL_relaxation_budget (0, 10.0)
RELAX_protein_ADA_pc (0, 0.0)
RELAX_protein_ADK_pc (0, 0.0)
RELAX_protein_AK1_pc (0, 0.0)
RELAX_protein_ALDOA_pc (0, 0.0)
RELAX_protein_ALDOB_pc (0, 0.0)
RELAX_protein_ALDOC_pc (0, 0.0)
RELAX_protein_AMPD3_pc (0, 0.0)
RELAX_protein_APRT_pc (0, 0.0)
RELAX_protein_AQP1_pc (0, 0.0)
RELAX_protein_AQP3_pc (0, 0.0)
RELAX_protein_ATP1A1_pc (0, 0.0)
RELAX_protein_ATP1A3_pc (0, 0.0)
RELAX_protein_ATP1A4_pc (0, 0.0)
RELAX_protein_ATP1B1_pc (0, 0.0)
RELAX_protein_ATP1B2_pc (0, 0.0)
RELAX_protein_ATP1B3_pc (0, 0.0)
RELAX_protein_ATP2B1_pc (0, 0.0)
RELAX_protein_ATP2B4_pc (0, 0.0)
RELAX_protein_BPGM_pc (0, 0.0)
RELAX_protein_BSG_pc (0, 0.0)
RELAX_protein_CACNA1A_pc (0, 0.0)
RELAX_protein_CYB5A_pc (0, 0.0)
RELAX_protein_CYB5B_pc (0, 0.0)
RELAX_protein_CYB5R3_pc (0, 0.0)
RELAX_protein_EMB_pc (0, 0.0)
RELAX_protein_ENO1_pc (0, 0.0)
RELAX_protein_ENO2_pc (0, 0.0)
REL

d:\Anaconda3\envs\RBC-GEM-akey7\Lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


<Solution infeasible at 0x1a956c47a70>

#### Generate results for subset of PC model reactions
##### Reactions necessary for all flux-expression correlation computations.
To reduce computation time, a subset of reactions can be defined. 
For flux-expression correlations, the minimum reaction set are reactions associated with genes associated and the corresponding enzyme dilution reaction for total enzyme.

In [ ]:
enzyme_total_suffix = DEFAULT_ENZYME_TOTAL_SUFFIX
min_reaction_list = model.reactions.query(lambda x: x.gene_reaction_rule).list_attr(
    "id"
)
enzymes_list = pcmodel.reactions.query(
    lambda x: x.id.startswith(f"ENZDL_enzyme_") and f"{enzyme_total_suffix}" in x.id
).list_attr("id")
reaction_enzymes_map = {
    rid: tuple(
        pcmodel.reactions.query(
            lambda x: x.id.startswith(f"ENZDL_enzyme_{rid}_")
        ).list_attr("id")
    )
    for rid in min_reaction_list
}
enzyme_reaction_map = {
    enzyme: rid for rid, enzymes in reaction_enzymes_map.items() for enzyme in enzymes
}
if not enzymes_list:
    enzymes_list = [
        enzyme
        for enzyme, rid in enzyme_reaction_map.items()
        if rid in min_reaction_list
    ]
min_reaction_list += enzymes_list
print(
    f"Number of reactions minimize/maximize (minimum): {len(min_reaction_list)} / {len(pcmodel.reactions)}"
)

In [7]:
pcmodel_dataset_parameterized.reactions.get_by_id(
    "PBDL_relaxation_budget"
).upper_bound = pcmodel_dataset_parameterized.reactions.get_by_id(
    "PBDL_relaxation_budget"
).lower_bound

In [ ]:
pcmodel_dataset_parameterized.optimize()
# pcmodel.objective.expression

## Run pcFVA

In [9]:
from cobra.flux_analysis import flux_variability_analysis

In [ ]:
verbose = True
optimum_percents = [0, 0.5, 0.9, 0.99]
reaction_list = min_reaction_list.copy()

pcfva_results_dirpath = Path(f"{dataset_path}/pcFVA")

index_cols = ["reactions", "optimum", "model"]
print("================================================")
print(f"Computing pcFVA results for {pcmodel_dataset_parameterized}")
print("================================================")
try:
    optimum_solutions = []
    if verbose:
        print(f"Starting simulations for {pcmodel_dataset_parameterized}")
    for optimum_percent in optimum_percents:
        pcfva_sol = flux_variability_analysis(
            pcmodel_dataset_parameterized,
            reaction_list=reaction_list,
            loopless=False,
            fraction_of_optimum=optimum_percent,
            processes=COBRA_CONFIGURATION.processes,
        )
        pcfva_sol.index = pd.MultiIndex.from_tuples(
            [
                (rid, optimum_percent, pcmodel_dataset_parameterized.id)
                for rid in pcfva_sol.index
            ],
            names=index_cols,
        )
        optimum_solutions.append(pcfva_sol)
        if verbose:
            print(f"Finished pcFVA for percent optimum: {optimum_percent}.")
    pcfva_sol = pd.concat(optimum_solutions, axis=0)
    filepath = Path(
        f"{pcfva_results_dirpath}/{pcmodel_dataset_parameterized}_FVAresults.tsv"
    )
    pcfva_sol.to_csv(filepath, sep="\t", index=True)
    pcfva_solutions[str(pcmodel_dataset_parameterized)] = pcfva_sol
    if verbose:
        print(f"Finished all simulations for {pcmodel_dataset_parameterized}")
except Exception as e:
    if verbose:
        print(f"{pcmodel_dataset_parameterized} failed due to an exception. {str(e)}\n")
    with open(f"{pcfva_results_dirpath}/pcFVA-errors.log", "a") as file:
        file.write(
            f"{pcmodel_dataset_parameterized} failed due to an exception. {str(e)}\n"
        )

In [ ]:
groupby_list = ["model", "reactions"]

# Initialize entries with prefixes used for seperating DataFrames
dict_of_dataframes_types = {
    "reactions": None,
    "proteins": "PROTDL",
    # "complexes": "CPLXFM",
    # "complex_dilutions": "CPLXDL",
    "enzymes": "ENZDL",
    # "enzyme_formation": "ENZFM",
    "budgets": "PBDL",
    "relaxation": "RELAX",
}
df_pcfva_all = pcfva_sol.reset_index(drop=False)
for key, prefix in dict_of_dataframes_types.copy().items():
    if prefix:
        df = df_pcfva_all[
            df_pcfva_all["reactions"].apply(lambda x: x.startswith(prefix))
        ]
    else:
        df = df_pcfva_all[
            df_pcfva_all["reactions"].apply(lambda x: x in model.reactions)
        ]
    dict_of_dataframes_types[key] = df.copy()

dict_of_dataframes_types

# Get the maximum value of the reaction flux in each direction, regardless of percent optimum
df = dict_of_dataframes_types["reactions"].copy()
df = df.groupby(groupby_list)[["minimum", "maximum"]].agg(
    {
        "minimum": "min",
        "maximum": "max",
    }
)
df_max_flux_per_model = df.abs().max(axis=1)
df_max_flux_per_model.name = "Flux"
df_max_flux_per_model

# Determine flux range
df = dict_of_dataframes_types["reactions"].copy()
df["Range"] = df["maximum"] - df["minimum"]
df_flux_range_per_model = df.groupby(groupby_list)["Range"].max()
df_flux_range_per_model

# Determine span association with reaction
df = dict_of_dataframes_types["enzymes"].copy()
df["reactions"] = df["reactions"].apply(lambda x: enzyme_reaction_map[x])
df_max_enzyme_per_model = df.groupby(groupby_list)["maximum"].max()
df_max_enzyme_per_model.name = "Expression"
df_max_enzyme_per_model

df_reaction_flux_expression = (
    pd.merge(
        df_max_flux_per_model,
        df_flux_range_per_model,
        left_index=True,
        right_index=True,
    )
    .merge(df_max_enzyme_per_model, left_index=True, right_index=True)
    .reset_index(drop=False)
)
df_reaction_flux_expression